# Iris Flower Classification
**OASIS INFOBYTE — Data Science Internship, Task 1**

**Author:** MAYUR R SHETTY  
**Track:** Data Science  
**Objective:** Predict Iris species (Setosa, Versicolor, Virginica) from sepal and petal measurements.

This notebook covers dataset inspection, EDA, feature discussion, an 80/20 split, three classifiers, evaluation, and model selection.

## 1. Import libraries and load the dataset
The Iris dataset is built into scikit-learn; no manual download is needed.

In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             ConfusionMatrixDisplay)

RANDOM_STATE = 42
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

iris = load_iris(as_frame=True)
X = iris.data.copy()
y = iris.target.copy()
target_names = list(iris.target_names)
df = X.copy()
df["species"] = y.map(dict(enumerate(target_names)))
print("Dataset shape:", df.shape)
display(df.head())

## 2. Data inspection
Inspect shape, types, missing values, duplicates, and descriptive statistics.

In [ ]:
print("Data types:")
display(df.dtypes.to_frame("dtype"))
print("Missing values:")
display(df.isna().sum().to_frame("missing_values"))
print("Duplicate rows:", int(df.duplicated().sum()))
display(df.describe().T)

## 3. Exploratory Data Analysis
Visualize class balance, feature relationships, and distributions.

In [ ]:
counts = df["species"].value_counts().reindex(target_names)
plt.figure(figsize=(7, 4))
plt.bar(counts.index, counts.values)
plt.title("Iris Species Distribution")
plt.xlabel("Species")
plt.ylabel("Number of samples")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "class_distribution.png", dpi=160)
plt.show()

In [ ]:
features = list(X.columns)
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for i, feature_x in enumerate(features):
    for j, feature_y in enumerate(features):
        ax = axes[i, j]
        if i == j:
            for species in target_names:
                vals = df.loc[df["species"] == species, feature_x]
                ax.hist(vals, alpha=0.55, label=species, bins=10)
        else:
            for species in target_names:
                subset = df[df["species"] == species]
                ax.scatter(subset[feature_y], subset[feature_x], alpha=0.7, s=12, label=species)
        if i == 3:
            ax.set_xlabel(feature_y.replace(" (cm)", ""))
        else:
            ax.set_xticklabels([])
        if j == 0:
            ax.set_ylabel(feature_x.replace(" (cm)", ""))
        else:
            ax.set_yticklabels([])
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper right")
fig.suptitle("Iris Feature Relationships by Species")
fig.tight_layout(rect=[0, 0, 0.9, 0.97])
fig.savefig(OUTPUT_DIR / "pairplot.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, feature in zip(axes.flat, X.columns):
    grouped = [df.loc[df["species"] == species, feature].values for species in target_names]
    ax.boxplot(grouped, labels=target_names)
    ax.set_title(feature.replace(" (cm)", "").title())
    ax.set_xlabel("")
fig.suptitle("Feature Distributions by Species")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "feature_boxplots.png", dpi=160)
plt.show()

### Feature selection discussion
Petal length and petal width are typically the most discriminative measurements: they separate Setosa clearly and help distinguish Versicolor from Virginica. Sepal measurements provide additional information but overlap more between classes. We retain all four features for a fair baseline comparison.

## 4. Train/test split
Use a stratified 80/20 split. Scaling is inside model pipelines to prevent test-set information leakage.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Training samples:", len(X_train), "| Testing samples:", len(X_test))
display(y_train.map(dict(enumerate(target_names))).value_counts())

## 5. Train three classifiers
Compare Logistic Regression, K-Nearest Neighbours, and Random Forest.

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    ]),
    "K-Nearest Neighbours": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=5))
    ]),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE)
}
results, fitted_models, predictions = {}, {}, {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    fitted_models[name], predictions[name] = model, pred
    results[name] = {
        "accuracy": accuracy_score(y_test, pred),
        "precision_macro": precision_score(y_test, pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_test, pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_test, pred, average="macro", zero_division=0)
    }
comparison = pd.DataFrame(results).T.sort_values(["accuracy", "f1_macro"], ascending=False)
display(comparison.round(4))

## 6. Evaluate the models
Use accuracy, macro precision, macro recall, macro F1, and confusion matrices.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, pred) in zip(axes, predictions.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_test, pred, display_labels=target_names, colorbar=False, ax=ax
    )
    ax.set_title(name)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "confusion_matrices.png", dpi=160)
plt.show()

for name, pred in predictions.items():
    print("\\n", name)
    print(classification_report(y_test, pred, target_names=target_names, zero_division=0))

In [ ]:
ax = comparison[["accuracy", "precision_macro", "recall_macro", "f1_macro"]].plot(
    kind="bar", figsize=(10, 5), ylim=(0, 1.05)
)
ax.set_title("Classifier Performance Comparison")
ax.set_ylabel("Score")
ax.set_xlabel("Model")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "model_comparison.png", dpi=160)
plt.show()

## 7. Select and save the best model
Choose the highest test accuracy; use macro F1 as a tie-breaker.

In [ ]:
best_name = max(results, key=lambda name: (results[name]["accuracy"], results[name]["f1_macro"]))
best_model = fitted_models[best_name]
best_pred = predictions[best_name]
print("Selected model:", best_name)
print(classification_report(y_test, best_pred, target_names=target_names, zero_division=0))
joblib.dump(best_model, OUTPUT_DIR / "best_iris_model.joblib")
payload = {
    "dataset_shape": list(df.shape),
    "features": list(X.columns),
    "classes": target_names,
    "train_samples": len(X_train),
    "test_samples": len(X_test),
    "model_metrics": comparison.to_dict(orient="index"),
    "best_model": best_name
}
(OUTPUT_DIR / "metrics.json").write_text(json.dumps(payload, indent=2), encoding="utf-8")
print("Saved outputs to:", OUTPUT_DIR.resolve())

## 8. Conclusion and limitations
The notebook compares three supervised classifiers and selects a model using measured hold-out performance. Petal length and width are especially informative. Iris is a small, clean educational dataset; results do not guarantee performance on arbitrary flowers. This model uses four numeric measurements, not photographs.

**Before submission:** MAYUR R SHETTY, run every cell from top to bottom, verify output files, and record your own end-to-end demo.